In [1]:
!hostname

gpua087.delta.ncsa.illinois.edu


In [3]:
# in a fresh notebook cell, BEFORE any other import
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
print(jax.devices())                       # should print [CpuDevice(...)]

import pickle
with open("/projects/bhdw/asachan/tmp/tp_atac_rna_skm.pkl", "rb") as f:
    tp = pickle.load(f)


[CpuDevice(id=0)]


In [4]:
import sys
print(sys.executable)

/projects/bhdw/asachan/.conda/envs/moscot/bin/python


In [5]:
import scanpy as sc
import scipy.sparse as sp
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import os
os.chdir('/projects/bgdb/asachan/methods/FIREFate/moscot')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
human_genome_path = '/work/hdd/bgdb/asachan/datasets_proj/human_genome_files'

out_tmp = '/projects/bhdw/asachan/tmp'

In [7]:
pd.set_option('mode.string_storage', 'python')

In [8]:
import moscot

### Load learnt coupling matrix between atac and rna and joint embeddings

In [9]:
joint = ad.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/joint_atac_rna_female_type2.h5ad')

In [10]:
adata_atac = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad')

In [11]:
# add age as categorical
adata_atac.obs["age_categorical"] = adata_atac.obs["age"].astype("category")

In [12]:
adata_rna = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2_ds_wrt_HALLMARK_DNA_REPAIR.h5ad')

#### Pull soft peak accessibility onto every RNA cell

In [37]:
import numpy as np, scipy.sparse as sp, anndata as ad

# stack the per-sample transport matrices into one (n_atac_total, n_rna) coupling
T_blocks, atac_obs_idx = [], []
for (src_b, tgt_b), prob in tp.problems.items():
    T_blocks.append(np.asarray(prob.solution.transport_matrix))   # (n_atac_b, n_rna)
    atac_obs_idx.append(adata_atac.obs.index[adata_atac.obs["sample"] == src_b])
T_full   = np.vstack(T_blocks)                                    # (n_atac_total, n_rna_ref)
atac_idx = np.concatenate(atac_obs_idx)
rna_idx  = adata_rna.obs.index                                    # tgt is the same "ref" set

# COLUMN-normalise so each RNA cell receives a probability distribution over ATAC cells
col_sum = T_full.sum(0, keepdims=True)
T_col   = T_full / np.where(col_sum > 0, col_sum, 1.0)            # (n_atac, n_rna)

# soft peak matrix per RNA cell:  X_peaks_rna[c] = sum_a T_col[a,c] * X_peaks_atac[a]
X_peak = adata_atac[atac_idx].X
X_peak = X_peak.toarray() if sp.issparse(X_peak) else X_peak
X_peaks_rna = T_col.T @ X_peak                                    # (n_rna, n_peaks)

adata_rna.obsm["X_peaks_inferred"] = X_peaks_rna.astype(np.float32)
print("inferred peaks per RNA cell:", X_peaks_rna.shape,
      "  mean nnz / cell:", float((X_peaks_rna > 0).sum(1).mean()))

inferred peaks per RNA cell: (3989, 27649)   mean nnz / cell: 27640.729756831286


In [38]:
print("col sum range:", col_sum.min(), col_sum.max())             # should all be ≈ 1.0/n_rna_ref

col sum range: 0.0007514826 0.0008067936


#### is the transition probability too diffused or does it still capture geometrical structure above uniform coupling?

In [39]:
import numpy as np

# T_full: (n_atac_total, n_rna_ref), already in memory
T_col = T_full / T_full.sum(0, keepdims=True)         # column-stochastic

top1_col   = T_col.max(0)                              # peak ATAC contributor per RNA cell
H_col      = -(T_col * np.log(T_col + 1e-30)).sum(0)
eff_k_col  = np.exp(H_col)                             # effective # ATAC cells contributing

print(f"per-RNA-cell ATAC concentration:")
print(f"  top1 mass     median={np.median(top1_col):.3f}  P95={np.percentile(top1_col,95):.3f}")
print(f"  eff. n ATAC   median={np.median(eff_k_col):.0f}  P5={np.percentile(eff_k_col,5):.0f}")
print(f"  total ATAC    {T_col.shape[0]}")

per-RNA-cell ATAC concentration:
  top1 mass     median=0.103  P95=0.332
  eff. n ATAC   median=119  P5=13
  total ATAC    5830


#### Multi-ome data

In [40]:
import muon as mu
# Import a module with ATAC-seq-related functions
from muon import atac as ac

In [41]:
adata_atac

AnnData object with n_obs × n_vars = 5830 × 27649
    obs: 'replicates', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'orig.ident', 'sample', 'group', 'ReadsInPeaks', 'FRIP', 'fiber_class_1_anno', 'Annotation', 'UMAP_1', 'UMAP_2', 'fiber_class_anno', 'country', 'age', 'Sex', 'age_categorical'
    var: 'n_cells'
    uns: 'Annotation_colors', 'lsi', 'neighbors', 'sample_colors', 'umap'
    obsm: 'X_UMAP', 'X_lsi', 'X_umap'
    varm: 'LSI'
    layers: 'Tn5_insertion_counts', 'tfidf'
    obsp: 'connectivities', 'distances'

In [42]:
# %% TF-IDF normalisation
# muon's tfidf writes the normalised matrix back into .X
ac.pp.tfidf(adata_atac, scale_factor=1e4)

In [43]:
#add it to layer 
adata_atac.layers["tfidf"] = adata_atac.X.copy()

In [44]:
import mudata as mu, anndata as ad, scipy.sparse as sp, numpy as np

# var = peaks, with chrom/start/end so SnapATAC2 can find motifs
peaks = adata_atac.var_names.to_list()
peak_var = adata_atac.var[["chrom", "start", "end"]].copy() if {"chrom","start","end"}.issubset(adata_atac.var.columns) else None

# fall back to parsing "chrN:S-E"
if peak_var is None:
    parsed = [p.replace(":", "-").split("-") for p in peaks]
    peak_var = (
        __import__("pandas").DataFrame(parsed, columns=["chrom", "start", "end"], index=peaks)
        .astype({"start": int, "end": int})
    )

    
# RNA modality (real expression)
rna = ad.AnnData(
    X    = adata_rna.X if sp.issparse(adata_rna.X) else sp.csr_matrix(adata_rna.X),
    obs  = adata_rna.obs.copy(),
    var  = adata_rna.var.copy(),
    obsm = {"X_pca": adata_rna.obsm["X_pca"]},
)

# ATAC modality (soft peaks pulled via OT, same RNA cells)
atac = ad.AnnData(
    X    = sp.csr_matrix(X_peaks_rna),                  # (n_rna_cells, n_peaks)
    obs  = adata_rna.obs.copy(),                        # SAME obs as rna → MuData aligns 1-to-1
    var  = peak_var,                                    # chrom/start/end indexed by peak id
)
atac.var_names = peaks

mome = mu.MuData({"rna": rna, "atac": atac})
print(mome)

MuData object with n_obs × n_vars = 3989 × 76004
  2 modalities
    rna:	3989 x 48355
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      obsm:	'X_pca'
    atac:	3989 x 27649
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'chrom', 'start', 'end'


/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [45]:
mome["rna"].var_names

Index(['WASH7P', 'CICP27', 'AL732372.2', 'AL669831.3', 'MTND2P28', 'MTATP6P1',
       'LINC01409', 'LINC01128', 'NOC2L', 'PERM1',
       ...
       'FAM8A3P', 'GRPEL2P3', 'MTCO1P8', 'AC010409.2', 'AL590635.1',
       'AC063955.1', 'MKNK2P1', 'AC087072.1', 'CDC42P2', 'CYCSP29'],
      dtype='string', length=48355)

#### Subset the rna to MaxToki vocab

In [46]:
import pandas as pd, pyranges as pr, numpy as np

# 1. symbol → ENSG from the same GTF you used for TSS lookup
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_name", "gene_id"]].drop_duplicates("gene_name")
gtf["gene_id_clean"] = gtf["gene_id"].str.split(".").str[0]    # strip ENSG version
sym2ensg = dict(zip(gtf["gene_name"].astype(str), gtf["gene_id_clean"]))
print(f"GTF mappings: {len(sym2ensg)} symbol→ENSG")

# 2. attach ENSG to mome["rna"].var
syms = mome["rna"].var_names.astype(str)
mome["rna"].var["ensg"] = [sym2ensg.get(s, None) for s in syms]

n_total   = mome["rna"].n_vars
n_mapped  = mome["rna"].var["ensg"].notna().sum()
print(f"{n_mapped}/{n_total} symbols mapped to ENSG  ({n_mapped/n_total:.1%})")

GTF mappings: 61471 symbol→ENSG
29718/48355 symbols mapped to ENSG  (61.5%)


In [47]:
import json, numpy as np, pandas as pd

# 1. load MaxToki vocab
with open("/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json") as f:
    vocab = json.load(f)

# strip special tokens to leave only ENSG entries
ensg_to_token = {k: v for k, v in vocab.items() if k.startswith("ENSG")}
print(f"vocab tokens: {len(vocab)} total, {len(ensg_to_token)} ENSG-keyed")

# 2. attach token index to mome["rna"].var
mome["rna"].var["maxtoki_token"] = (
    mome["rna"].var["ensg"].map(ensg_to_token).astype("Int64")  # nullable int
)

n_in_vocab = mome["rna"].var["maxtoki_token"].notna().sum()
print(f"vars with MaxToki token: {n_in_vocab} / {mome['rna'].n_vars} "
      f"({n_in_vocab / mome['rna'].n_vars:.1%})")

vocab tokens: 20277 total, 20271 ENSG-keyed
vars with MaxToki token: 18470 / 48355 (38.2%)


In [48]:
import numpy as np

mask = mome["rna"].var["maxtoki_token"].notna()
print(f"keeping {mask.sum()} / {mome['rna'].n_vars} genes")

# rebuild MuData with subsetted RNA modality, ATAC unchanged
import mudata as mu
rna_sub = mome["rna"][:, mask.values].copy()
rna_sub.var["maxtoki_token"] = rna_sub.var["maxtoki_token"].astype(int)
rna_sub = rna_sub[:, np.argsort(rna_sub.var["maxtoki_token"].values)].copy()  # sort by token id

mome = mu.MuData({"rna": rna_sub, "atac": mome["atac"]})
print(mome)

keeping 18470 / 48355 genes
MuData object with n_obs × n_vars = 3989 × 46119
  2 modalities
    rna:	3989 x 18470
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensg', 'maxtoki_token'
      obsm:	'X_pca'
    atac:	3989 x 27649
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'chrom', 'start', 'end'


/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [49]:
#overwrite mome file 
mome.write_h5mu(f"{out_tmp}/mome_atac_rna_skm.h5mu")

/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


# Peak to gene (TSS +- 1kb & TF-Motif & OCR counts)

#### TSS finder

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, pyranges as pr
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

# --- 1. ENSG-keyed TSS lookup -------------------------------------------
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_id", "Chromosome", "Start", "End", "Strand"]]
gtf["TSS"]     = np.where(gtf.Strand == "+", gtf.Start, gtf.End)
gtf["gene_id"] = gtf["gene_id"].str.split(".").str[0]                  # strip ENSG version
tss = gtf.drop_duplicates("gene_id").set_index("gene_id")[["Chromosome", "TSS"]]
print(f"{len(tss):,} ENSG → TSS")

# --- 2. modality views --------------------------------------------------
atac    = mome["atac"]
rna     = mome["rna"]
genes_e = rna.var["ensg"].astype(str).values                            # ENSG, sorted by token
peaks   = atac.var_names.to_list()

peak_mid = ((atac.var["start"].astype(int) + atac.var["end"].astype(int)) // 2).values
peak_chr = atac.var["chrom"].astype(str).values

# --- 3. vectorised candidate pairs in ±100 kb (chromosome-grouped) ------
WIN = 100_000   # SCARLINK uses ±250 kb to capture full CRE range; 100 kb is conservative

# group peak indices by chromosome for fast lookup
peak_by_chr = {}
for ch in np.unique(peak_chr):
    peak_by_chr[ch] = (np.where(peak_chr == ch)[0], peak_mid[peak_chr == ch])

# look up TSS per gene; accept only if ENSG present in gtf
present = np.isin(genes_e, tss.index.values)
print(f"{present.sum():,}/{len(genes_e):,} genes have TSS in gtf")

cand_p, cand_g = [], []
for gi in tqdm(np.where(present)[0], desc="peak-gene candidates"):
    e        = genes_e[gi]
    chrom, t = tss.at[e, "Chromosome"], int(tss.at[e, "TSS"])
    if chrom not in peak_by_chr: continue
    p_idx, p_mid = peak_by_chr[chrom]
    near = p_idx[np.abs(p_mid - t) <= WIN]
    if near.size:
        cand_p.append(near)
        cand_g.append(np.full(near.size, gi, dtype=np.int64))

cand_p = np.concatenate(cand_p)
cand_g = np.concatenate(cand_g)
print(f"{len(cand_p):,} peak–gene candidates")

# --- 4. vectorised Pearson r across cells -------------------------------
def zscore(M):
    M = M.toarray() if sp.issparse(M) else np.asarray(M)
    return (M - M.mean(0)) / (M.std(0) + 1e-8)

peak_mat = atac.layers["tfidf"] if "tfidf" in atac.layers else atac.X
Zp = zscore(peak_mat)                # (n_cells, n_peaks)
Zg = zscore(rna.X)                   # (n_cells, n_genes_subset)
n  = Zp.shape[0]

r = (Zp[:, cand_p] * Zg[:, cand_g]).sum(0) / n
keep = np.abs(r) > 0.05
print(f"{keep.sum():,} significant links (|r|>0.05)")

# --- 5. assemble sparse P2G ---------------------------------------------
P2G = csr_matrix(
    (r[keep].astype(np.float32), (cand_p[keep], cand_g[keep])),
    shape=(atac.n_vars, rna.n_vars),
)
print("P2G:", P2G.shape, "nnz:", P2G.nnz)

sp.save_npz(f"{out_tmp}/P2G.npz", P2G)

63,086 ENSG → TSS
18,470/18,470 genes have TSS in gtf


peak-gene candidates:   0%|          | 0/18470 [00:00<?, ?it/s]

91,500 peak–gene candidates
6,697 significant links (|r|>0.05)
P2G: (27649, 18470) nnz: 6697


In [51]:
links_per_gene = (P2G != 0).sum(0).A1
print(f"genes with ≥1 link : {(links_per_gene > 0).sum()} / {P2G.shape[1]}")
print(f"  median links/gene (among linked): {np.median(links_per_gene[links_per_gene > 0]):.0f}")
print(f"  P95 links/gene  : {np.percentile(links_per_gene, 95):.0f}")

genes with ≥1 link : 2868 / 18470
  median links/gene (among linked): 1
  P95 links/gene  : 2


### TF Motif Scan per peak

# Build bf16 attention bias

In [ ]:
import torch, numpy as np

def build_attention_bias(
    open_c,                  # (n_peaks,) bf16  — accessibility for ONE cell
    token_ids,               # (L,) int64       — vocab tokens for this cell's sequence
    P2G_t, MOTIF_t,          # sparse bf16     — (n_peaks, n_genes), (n_peaks, n_motifs)
    motif_token_arr,         # (n_motifs,)     int64, -1 if motif's TF has no vocab token
    gene_token_arr,          # (n_genes,)      int64
    lam=1.0,
):
    # 1. per-cell GRN  G_c[motif, gene] = (MOTIF * open).T @ P2G
    weighted = MOTIF_t * open_c.unsqueeze(1)                    # (n_peaks, n_motifs)
    G_c      = torch.sparse.mm(weighted.T, P2G_t).to_dense()    # (n_motifs, n_genes)

    # 2. column subset: gather G_c columns by token_ids
    L         = token_ids.shape[0]
    inv_gene  = torch.full((int(gene_token_arr.max())+2,), -1,
                           dtype=torch.long, device=G_c.device)
    inv_gene[gene_token_arr] = torch.arange(len(gene_token_arr), device=G_c.device)
    cols      = inv_gene[token_ids]                              # (L,) local gene idx, -1 if absent
    G_seq     = torch.where(
        (cols >= 0).unsqueeze(0),
        G_c[:, cols.clamp(min=0)],
        torch.zeros_like(G_c[:, :1]).expand(-1, L),
    )                                                            # (n_motifs, L)

    # 3. row subset + max-pool by TF token
    valid     = motif_token_arr >= 0
    motif_tok = motif_token_arr[valid]                           # (n_valid,)
    G_valid   = G_seq[valid]                                     # (n_valid, L)

    # which sequence positions are TF tokens, indexed by motif row?
    # bias[:, j] gets the max over motifs whose TF token == token_ids[j]
    bias      = torch.zeros(L, L, dtype=torch.bfloat16, device=G_c.device)
    is_tf_pos = (token_ids.unsqueeze(0) == motif_tok.unsqueeze(1))   # (n_valid, L)
    # for each j with at least one matching motif, bias[:, j] = max over rows where is_tf_pos[:, j]
    contrib   = G_valid.unsqueeze(2) * is_tf_pos.unsqueeze(2).float()  # (n_valid, L_query=identical, L)
    # actually we want: bias[i, j] = max_m G_valid[m, i] * is_tf_pos[m, j]
    bias_f32  = (G_valid.unsqueeze(2) * is_tf_pos.float().unsqueeze(1)).amax(dim=0)  # (L, L)

    return lam * torch.log1p(bias_f32).to(torch.bfloat16)